# IMPORTS

In [86]:
import glob
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks
import numpy as np
from sklearn import tree
from scipy.ndimage import gaussian_filter1d
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.tree import DecisionTreeClassifier,export_graphviz
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from scipy.interpolate import interp1d
import neurokit2 as nk

# FILTERS

In [87]:
def butter_lowpass_filter(data, cutoff, fs, order=2):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = filtfilt(b, a, data)
    return y

def butter_highpass_filter(data, cutoff, fs, order=2):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='highpass', analog=False)
    y = filtfilt(b, a, data)
    return y

def bandpass_filter(data, lowcut, highcut, fs, order=2):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return filtfilt(b, a, data)

# Upsampling signals

Upsample the signal to new sampling rate fs_new.

time_orig: original time vector in seconds (1D array or Series)

signal_orig: original signal values (1D array or Series)

fs_new: target sampling rate in Hz

## Returns:

time_new: new time vector at fs_new

signal_new: interpolated signal at time_new

In [88]:
def upsample_signal(time_orig, signal_orig, fs_new=15):

    duration = time_orig[-1] - time_orig[0]
    n_samples_new = int(duration * fs_new) + 1

    time_new = np.linspace(time_orig[0], time_orig[-1], n_samples_new)

    # Interpolator
    interpolator = interp1d(time_orig, signal_orig, kind='linear', fill_value="extrapolate")

    signal_new = interpolator(time_new)

    return time_new, signal_new

# EA Processing

In [89]:
def ea_detection(csv_file_path, fs=15):
    df = pd.read_csv(csv_file_path)
    df['time[s]'] = (df['LocalTimestamp'] - df['LocalTimestamp'].iloc[0])
    df = df.loc[(df['time[s]'] > 120) & (df['time[s]'] < (df['time[s]'].iloc[-1]) - 120)]

    ea_raw = df['EA'].astype(float)

    # Bandpass filter at original sampling rate
    ea_filtered = bandpass_filter(ea_raw, 0.1, 5, fs)
    time_ea = df['time[s]'].values


    return time_ea, ea_filtered



# HR Feature Detection

In [90]:
def hr_features_from_window(hr_window):
    hr_window = hr_window.reset_index(drop=True)
    if len(hr_window) < 2:
        # Not enough samples, return zeros
        return {key: 0 for key in [
            'mean_hr', 'median_hr', 'std_hr', 'min_hr', 'max_hr',
            'minRatio_hr', 'maxRatio_hr', 'median_first_derivative',
            'min_first_derivative', 'max_first_derivative',
            'minRatio_first_derivative', 'maxRatio_first_derivative',
            'std_first_derivative', 'min_second_derivative',
            'max_second_derivative', 'std_second_derivative',
            'minRatio_second_derivative', 'maxRatio_second_derivative'
        ]}

    first_diff = hr_window.diff().dropna()
    second_diff = first_diff.diff().dropna()

    def safe_ratio(a, b):
        return a / b if b != 0 else 0

    features = {
        'mean_hr': hr_window.mean(),
        'median_hr': hr_window.median(),
        'std_hr': hr_window.std(),
        'min_hr': hr_window.min(),
        'max_hr': hr_window.max(),
        'minRatio_hr': safe_ratio(hr_window.min(), hr_window.max()),
        'maxRatio_hr': safe_ratio(hr_window.max(), hr_window.min()),
        'median_first_derivative': first_diff.median(),
        'min_first_derivative': first_diff.min(),
        'max_first_derivative': first_diff.max(),
        'minRatio_first_derivative': safe_ratio(first_diff.min(), first_diff.max()),
        'maxRatio_first_derivative': safe_ratio(first_diff.max(), first_diff.min()),
        'std_first_derivative': first_diff.std(),
        'min_second_derivative': second_diff.min(),
        'max_second_derivative': second_diff.max(),
        'std_second_derivative': second_diff.std(),
        'minRatio_second_derivative': safe_ratio(second_diff.min(), second_diff.max()),
        'maxRatio_second_derivative': safe_ratio(second_diff.max(), second_diff.min())
    }
    return features

# Windowed Feature Extraction

In [91]:
def windowed_feature_extraction(time_ea, ea_signal, time_hr, hr_signal, window_sec=5, fs=15, overlap=0.5, label='unknown'):
    window_size = int(window_sec * fs)  # samples per window
    step_size = int(window_size * (1 - overlap))  # step size between windows
    n_samples = len(ea_signal)

    features_list = []
    
    # Create HR dataframe for easy masking
    df_hr = pd.DataFrame({'time[s]': time_hr, 'HR': hr_signal})

    for start in range(0, n_samples - window_size + 1, step_size):
        end = start + window_size

        ea_win = ea_signal[start:end]
        time_win = time_ea[start:end]

        # Process EDA window and extract EDA features
        signals, _ = nk.eda_process(ea_win, sampling_rate=fs, method='neurokit')

        eda_features = {
            'SCR_Onsets_sum': signals['SCR_Onsets'].sum(),
            'SCR_Peaks_sum': signals['SCR_Peaks'].sum(),
            'SCR_Height_mean': signals['SCR_Height'].mean() if len(signals) > 0 else 0,
            'SCR_Amplitude_mean': signals['SCR_Amplitude'].mean() if len(signals) > 0 else 0,
            'SCR_RiseTime_mean': signals['SCR_RiseTime'].mean() if len(signals) > 0 else 0,
            'SCR_Recovery_mean': signals['SCR_Recovery'].mean() if len(signals) > 0 else 0,
            'SCR_RecoveryTime_mean': signals['SCR_RecoveryTime'].mean() if len(signals) > 0 else 0,
        }

        # Get corresponding HR values for same time window (tolerance +/- small epsilon)
        hr_window = df_hr[(df_hr['time[s]'] >= time_win[0]) & (df_hr['time[s]'] <= time_win[-1])]['HR']

        # Skip windows with insufficient HR data
        if len(hr_window) < window_size * 0.8:  # 80% coverage threshold
            continue

        hr_feats = hr_features_from_window(hr_window)

        combined_features = {**eda_features, **hr_feats, 'label': label}

        features_list.append(combined_features)

    return pd.DataFrame(features_list)

# ML Model

In [92]:
def pred_tree(frames):
    X = frames.drop('label', axis=1) #drops 'label' column
    y = frames['label']

    #splits into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    dt_model = DecisionTreeClassifier(criterion='entropy', max_depth=3)

    #trains model?
    dt_model.fit(X_train, y_train)

    #predicts 'y' values with test 'x' values
    y_pred = dt_model.predict(X_test)

    #checks accuracy of predicted 'y' against true 'y'
    acc = accuracy_score(y_test, y_pred)

    # confusion matrix
    dt_cm = confusion_matrix(y_test, y_pred, labels=dt_model.classes_)

    # precision, recall, f1 score
    print(classification_report(y_test, y_pred))

    print("Decision Tree Accuracy:", acc)

    print("--------------------------------------------")

    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_test)

    print(classification_report(y_test, rf_pred))
    print("Random Forest Accuracy:", rf_model.score(X_test, y_test))


# Code to Print Graphs:

In [93]:
# def graph(raw_data):
#     df_min = raw_data["LocalTimestamp"].min() + 200
#     df_max = raw_data["LocalTimestamp"].max() - 200
#     df = raw_data[(raw_data["LocalTimestamp"]>df_min) & (raw_data["LocalTimestamp"]<df_max)]

#     df_ea = df["EA"]

#     nk.signal_plot(df_ea, sampling_rate=15) # Check basic graph

# Running Program . . . 

In [94]:
# Run detection

fs_eda = 15
window_sec = 5
overlap = 0.5

data = pd.DataFrame()

# Engaged files
engaged_filenames = glob.glob("engaged_EA/*.csv")
for file in engaged_filenames:
    time_ea, ea_filtered = ea_detection(file, fs=fs_eda)

    x = file.split("_")
    y = x[1].split("\\")
    hr_name = "engaged_HR" + "\\" + y[1] + "_" + x[2] + "_HR.csv"

    df_hr = pd.read_csv(hr_name)
    df_hr['time[s]'] = (df_hr['LocalTimestamp'] - df_hr['LocalTimestamp'].iloc[0])

    # Upsample HR to 15 Hz to align with EDA
    time_hr_up, hr_up = upsample_signal(df_hr['time[s]'].values, df_hr['HR'].values, fs_new=fs_eda)

    windowed_df = windowed_feature_extraction(time_ea, ea_filtered, time_hr_up, hr_up,
                                                window_sec=window_sec, fs=fs_eda,
                                                overlap=overlap, label='engaged')
    
    print(f"engaged file {file} generated {len(windowed_df)} windowed samples")

    if not windowed_df.empty:
        data = pd.concat([data, windowed_df], ignore_index=True)
    else:
        print(f"Warning: No valid windows extracted from engaged file {file}")

    data = pd.concat([data, windowed_df], ignore_index=True)

# Relaxed files
relaxed_filenames = glob.glob("relaxed_EA/*.csv")
for file in relaxed_filenames:
    time_ea, ea_filtered = ea_detection(file, fs=fs_eda)

    x = file.split("_")
    y = x[1].split("\\")
    hr_name = "relaxed_HR" + "\\" + y[1] + "_" + x[2] + "_HR.csv"

    df_hr = pd.read_csv(hr_name)
    df_hr['time[s]'] = (df_hr['LocalTimestamp'] - df_hr['LocalTimestamp'].iloc[0])

    # Upsample HR to 15 Hz to align with EDA
    time_hr_up, hr_up = upsample_signal(df_hr['time[s]'].values, df_hr['HR'].values, fs_new=fs_eda)

    # Extract features with windowing
    windowed_df = windowed_feature_extraction(time_ea, ea_filtered, time_hr_up, hr_up,
                                                window_sec=window_sec, fs=fs_eda,
                                                overlap=overlap, label='relaxed')
    
    print(f"engaged file {file} generated {len(windowed_df)} windowed samples")

    if not windowed_df.empty:
        data = pd.concat([data, windowed_df], ignore_index=True)
    else:
        print(f"Warning: No valid windows extracted from engaged file {file}")

    data = pd.concat([data, windowed_df], ignore_index=True)

print(f"Total samples: {len(data)}")
pred_tree(data)

C:\Users\Administrator\AppData\Roaming\Python\Python312\site-packages\neurokit2\eda\eda_peaks.py:127: RuntimeWarning: All-NaN slice encountered
  info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"])


engaged file engaged_EA\2025-07-10_15-09-52-005859_EA.csv generated 143 windowed samples
engaged file engaged_EA\2025-07-15_18-53-07-705441_EA.csv generated 145 windowed samples
engaged file engaged_EA\2025-07-15_19-32-05-631421_EA.csv generated 156 windowed samples
engaged file engaged_EA\2025-07-16_18-00-15-608675_EA.csv generated 282 windowed samples
engaged file engaged_EA\2025-07-17_15-27-24-195986_EA.csv generated 278 windowed samples


C:\Users\Administrator\AppData\Roaming\Python\Python312\site-packages\neurokit2\eda\eda_peaks.py:127: RuntimeWarning: All-NaN slice encountered
  info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"])


engaged file engaged_EA\2025-07-17_17-29-34-069561_EA.csv generated 279 windowed samples
engaged file engaged_EA\2025-07-17_19-16-34-115458_EA.csv generated 272 windowed samples
engaged file relaxed_EA\2025-07-10_14-37-42-966335_EA.csv generated 204 windowed samples
engaged file relaxed_EA\2025-07-15_18-09-31-471018_EA.csv generated 168 windowed samples
engaged file relaxed_EA\2025-07-15_19-12-47-299953_EA.csv generated 147 windowed samples
engaged file relaxed_EA\2025-07-16_17-25-19-561692_EA.csv generated 267 windowed samples


C:\Users\Administrator\AppData\Roaming\Python\Python312\site-packages\neurokit2\eda\eda_peaks.py:127: RuntimeWarning: All-NaN slice encountered
  info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"])


engaged file relaxed_EA\2025-07-17_14-56-58-835144_EA.csv generated 276 windowed samples
engaged file relaxed_EA\2025-07-17_16-34-08-884190_EA.csv generated 276 windowed samples
engaged file relaxed_EA\2025-07-17_17-52-10-833046_EA.csv generated 274 windowed samples
Total samples: 6334
              precision    recall  f1-score   support

     engaged       0.77      0.74      0.76       647
     relaxed       0.74      0.77      0.76       620

    accuracy                           0.76      1267
   macro avg       0.76      0.76      0.76      1267
weighted avg       0.76      0.76      0.76      1267

Decision Tree Accuracy: 0.7561168113654302
--------------------------------------------
              precision    recall  f1-score   support

     engaged       0.97      0.96      0.96       647
     relaxed       0.96      0.97      0.96       620

    accuracy                           0.96      1267
   macro avg       0.96      0.96      0.96      1267
weighted avg       0.96   